## BASIC DATA CHECK

In [4]:
# ================================
# BASIC DATA CHECK
# ================================
import numpy as np
import pandas as pd
df = pd.read_csv('realtraffic.csv')
print("Shape of dataset:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSample Data:")
print(df.head())

# ================================
# TARGET CHECK
# ================================

# If binary label already exists
if 'binary_label' in df.columns:
    print("\nBinary Label Distribution:")
    print(df['binary_label'].value_counts())

else:
    print("\nCreating Binary Label...")
    df['binary_label'] = df['attack_type_label'].apply(
        lambda x: 0 if x == 'normal' else 1
    )
    
    print("\nBinary Label Distribution:")
    print(df['binary_label'].value_counts())

# ================================
# CLASS BALANCE %
# ================================
print("\nClass Distribution (%):")
print(df['binary_label'].value_counts(normalize=True) * 100)

# ================================
# NUMERIC SUMMARY
# ================================
print("\nStatistical Summary:")
print(df.describe())

Shape of dataset: (13000, 80)

Columns:
Index(['timestamp', 'source_ip', 'destination_ip', 'protocol', 'http_method',
       'url', 'request_headers', 'request_body', 'response_code',
       'response_size', 'response_headers', 'user_agent', 'payload_raw',
       'attack_type_label', 'hour', 'day_of_week', 'is_weekend', 'is_attack',
       'parsed_request_headers', 'parsed_response_headers', 'Host',
       'User-Agent', 'Accept', 'Content-Type', 'Server', 'url_length',
       'url_num_params', 'url_num_special_chars', 'url_has_sql_keywords',
       'url_has_xss_patterns', 'url_has_cmd_injection', 'url_has_traversal',
       'url_has_ssrf_patterns', 'url_has_typosquatting', 'url_contains_ip',
       'url_has_credential_patterns', 'payload_length', 'request_body_length',
       'payload_special_chars', 'payload_num_sql_keywords',
       'payload_num_xss_patterns', 'payload_has_encoding', 'payload_is_empty',
       'payload_contains_hex', 'combined_payload', 'response_code_group',
       

###  DROP LEAKAGE + IRRELEVANT COLUMNS

In [5]:
# ================================
# DROP LEAKAGE + IRRELEVANT COLUMNS
# ================================
drop_cols = [
    'attack_type_label', 'is_attack',

    'attack_pattern_sql', 'attack_pattern_xss', 'attack_pattern_cmd',
    'attack_pattern_traversal', 'attack_pattern_ssrf',
    'attack_pattern_typosquatting', 'attack_pattern_credential',

    'source_ip_attack_ratio', 'destination_ip_attack_count',
    'is_known_attacker_ip',

    'timestamp', 'source_ip', 'destination_ip', 'url',
    'request_headers', 'response_headers', 'payload_raw',
    'combined_payload', 'parsed_request_headers', 'parsed_response_headers',
    'timestamp_min'
]

df_clean = df.drop(columns=[col for col in drop_cols if col in df.columns])

# ================================
# HANDLE MISSING VALUES
# ================================
df_clean = df_clean.fillna(0)

# ================================
# FIX NEGATIVE / OUTLIER VALUES
# ================================
num_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns

for col in num_cols:
    df_clean[col] = np.clip(df_clean[col],
                           df_clean[col].quantile(0.01),
                           df_clean[col].quantile(0.99))

# ================================
# FINAL FEATURES + TARGET
# ================================
X = df_clean.drop(columns=['binary_label'])
y = df_clean['binary_label']

print("Final Shape:", X.shape)

Final Shape: (13000, 57)


## BinaryClassification 
### Random Forest 

In [11]:
# Modified Random Forest Model - Addressing Potential Data Leakage
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('realtraffic.csv')

# Define features to exclude - ADDED SUSPICIOUS FEATURES
exclude_cols = ['timestamp', 'is_attack', 'attack_type_label', 'is_attack', 
                'url', 'request_headers', 'response_headers', 'user_agent', 
                'payload_raw', 'parsed_request_headers', 'parsed_response_headers',
                'Host', 'User-Agent', 'Accept', 'Content-Type', 'combined_payload',
                'timestamp_min',
                # REMOVING POTENTIALLY LEAKING FEATURES
                'payload_length',           # Too predictive - may leak label
                'response_payload_ratio',   # Strongly correlated with attack
                'payload_is_empty',         # Direct indicator
                'response_size_ratio',      # Correlated with payload
                'payload_special_chars',    # Attack indicator
                'payload_num_sql_keywords', # Direct attack indicator
                'payload_num_xss_patterns', # Direct attack indicator
                'payload_has_encoding',     # Attack pattern indicator
                'payload_contains_hex',     # Attack pattern indicator
                'combined_payload'          # Contains raw attack payload
                ]

# Select features
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Features used for modeling: {len(feature_cols)}")
print(f"Features excluded: {len(exclude_cols)}")
print("\nRemaining features:", feature_cols[:15], "...")

# Prepare features and target
X = df[feature_cols].copy()
y = df['is_attack'].copy()

# Handle categorical features
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Handle any remaining non-numeric values
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].astype(str).apply(lambda x: hash(x) % 1000)

# Convert all to numeric and handle infinities
X = X.apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest Model with Cross-Validation
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, 
                                   min_samples_split=20, min_samples_leaf=10,
                                   random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Cross-validation to check generalization
cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, 
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                             scoring='accuracy')

# Evaluation Metrics
print("\n" + "="*60)
print("MODIFIED RANDOM FOREST MODEL RESULTS")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"\nCross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features (After Removing Suspicious Ones):")
print(feature_importance.head(10).to_string(index=False))

# Check for remaining potential leakage
print("\n" + "="*60)
print("DIAGNOSTIC CHECK")
print("="*60)
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Class distribution in training - Normal: {sum(y_train==0)}, Attack: {sum(y_train==1)}")
print(f"Class distribution in test - Normal: {sum(y_test==0)}, Attack: {sum(y_test==1)}")

Features used for modeling: 55
Features excluded: 27

Remaining features: ['source_ip', 'destination_ip', 'protocol', 'http_method', 'request_body', 'response_code', 'response_size', 'hour', 'day_of_week', 'is_weekend', 'Server', 'url_length', 'url_num_params', 'url_num_special_chars', 'url_has_sql_keywords'] ...

MODIFIED RANDOM FOREST MODEL RESULTS
Accuracy: 0.9658
Precision: 0.9790
Recall: 0.9479
F1-Score: 0.9632

Cross-Validation Accuracy: 0.9633 (+/- 0.0050)

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.98      0.97      1372
      Attack       0.98      0.95      0.96      1228

    accuracy                           0.97      2600
   macro avg       0.97      0.96      0.97      2600
weighted avg       0.97      0.97      0.97      2600


Confusion Matrix:
[[1347   25]
 [  64 1164]]

Top 10 Most Important Features (After Removing Suspicious Ones):
            feature  importance
  is_attack_tool_ua    0.162928
     

In [14]:
# Improved Logistic Regression Model with Stronger Regularization
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('realtraffic.csv')

# Use is_attack as target
y = df['is_attack'].copy()

print("Target Variable Distribution (is_attack):")
print(f"Normal (0): {sum(y==0)}")
print(f"Attack (1): {sum(y==1)}")
print(f"Attack percentage: {sum(y==1)/len(y)*100:.2f}%\n")

# Define features to exclude (more aggressive exclusion)
exclude_cols = ['timestamp', 'binary_label', 'attack_type_label', 'is_attack', 
                'url', 'request_headers', 'response_headers', 'user_agent', 
                'payload_raw', 'parsed_request_headers', 'parsed_response_headers',
                'Host', 'User-Agent', 'Accept', 'Content-Type', 'combined_payload',
                'timestamp_min',
                # Removing ALL payload-related features
                'payload_length', 'response_payload_ratio', 'payload_is_empty',
                'response_size_ratio', 'payload_special_chars',
                'payload_num_sql_keywords', 'payload_num_xss_patterns',
                'payload_has_encoding', 'payload_contains_hex', 'combined_payload',
                'payload_length_safe',  # Also remove this one
                # Remove highly correlated attack pattern features
                'attack_pattern_sql', 'attack_pattern_xss', 'attack_pattern_cmd',
                'attack_pattern_traversal', 'attack_pattern_ssrf',
                'attack_pattern_typosquatting', 'attack_pattern_credential'
                ]

# Select features
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"Features used for modeling: {len(feature_cols)}")
print(f"Features excluded: {len(exclude_cols)}\n")
print("Remaining features:", feature_cols[:20])

# Prepare features
X = df[feature_cols].copy()

# Handle categorical features
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Handle any remaining non-numeric values
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].astype(str).apply(lambda x: hash(x) % 1000)

# Convert all to numeric and handle infinities
X = X.apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

# Remove features with near-zero variance
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)
selected_features = X.columns[selector.get_support()].tolist()
X = X[selected_features]
print(f"\nFeatures after variance threshold: {len(selected_features)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression with stronger regularization
# Try different C values (smaller C = stronger regularization)
C_values = [0.01, 0.05, 0.1, 0.5, 1.0]
best_c = 0.1
best_score = 0

for c in C_values:
    lr_test = LogisticRegression(C=c, max_iter=1000, random_state=42, 
                                 class_weight='balanced', solver='liblinear')
    cv_scores = cross_val_score(lr_test, X_train_scaled, y_train, 
                                 cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                                 scoring='accuracy')
    mean_score = cv_scores.mean()
    if mean_score > best_score:
        best_score = mean_score
        best_c = c

print(f"\nBest C value from cross-validation: {best_c}")

# Train final model with best C
lr_model = LogisticRegression(C=best_c, max_iter=1000, random_state=42, 
                              class_weight='balanced', solver='liblinear')
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = lr_model.predict(X_test_scaled)
y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

# Cross-validation
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, 
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
                             scoring='accuracy')

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Evaluation Metrics
print("\n" + "="*60)
print("IMPROVED LOGISTIC REGRESSION MODEL RESULTS")
print("="*60)
print(f"Regularization Strength (C): {best_c}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"\nCross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Feature Importance (coefficients)
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'coefficient': lr_model.coef_[0],
    'abs_coefficient': np.abs(lr_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

print("\nTop 15 Most Important Features (by coefficient magnitude):")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(feature_importance.head(15)[['feature', 'coefficient']].to_string(index=False))

# Check if coefficients are now more reasonable
max_coef = feature_importance.head(1)['abs_coefficient'].values[0]
print(f"\nMaximum coefficient magnitude: {max_coef:.4f}")
if max_coef > 2.0:
    print("⚠️  Warning: Coefficients still high - consider even stronger regularization")
else:
    print("✓ Coefficients are within reasonable range")

# Calculate training vs test performance gap
train_pred = lr_model.predict(X_train_scaled)
train_accuracy = accuracy_score(y_train, train_pred)

print("\n" + "="*60)
print("DIAGNOSTIC CHECK")
print("="*60)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Gap: {train_accuracy - accuracy_score(y_test, y_pred):.4f}")

# Check feature correlations with target
print("\nTop 5 Features with highest correlation to target:")
correlations = X_train.corrwith(pd.Series(y_train)).abs().sort_values(ascending=False)
print(correlations.head(5))

Target Variable Distribution (is_attack):
Normal (0): 6859
Attack (1): 6141
Attack percentage: 47.24%

Features used for modeling: 47
Features excluded: 35

Remaining features: ['source_ip', 'destination_ip', 'protocol', 'http_method', 'request_body', 'response_code', 'response_size', 'hour', 'day_of_week', 'is_weekend', 'Server', 'url_length', 'url_num_params', 'url_num_special_chars', 'url_has_sql_keywords', 'url_has_xss_patterns', 'url_has_cmd_injection', 'url_has_traversal', 'url_has_ssrf_patterns', 'url_has_typosquatting']

Features after variance threshold: 42

Best C value from cross-validation: 1.0

IMPROVED LOGISTIC REGRESSION MODEL RESULTS
Regularization Strength (C): 1.0
Accuracy: 0.9573
Precision: 0.9643
Recall: 0.9446
F1-Score: 0.9543
ROC-AUC Score: 0.9761

Cross-Validation Accuracy: 0.9540 (+/- 0.0067)

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.97      0.96      1372
      Attack       0.96      0.94      